# Model Serialization and Inference Preparation

This notebook prepares the final Logistic Regression model for production use. The final pipeline is rebuilt using the selected reduced feature set, trained on the training data, serialized to disk, and verified after loading.

In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv(
    "../data/processed/telco_customer_churn_cleaned.csv"
)


In [3]:
reduced_features = [
    "Senior Citizen",
    "Partner",
    "Dependents",
    "Tenure Months",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Contract",
    "Paperless Billing",
    "Payment Method",
    "Monthly Charges",
    "Total Charges"
]

In [4]:
X = df[reduced_features]
y = df["Churn Label"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
numeric_features = [
    "Tenure Months",
    "Monthly Charges",
    "Total Charges"
]

categorical_features = [
    feature
    for feature in reduced_features
    if feature not in numeric_features
]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

In [6]:
final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        C=1,
        penalty="l1",
        solver="liblinear",
        max_iter=1000,
        random_state=42
    ))
])

In [ ]:
final_pipeline.fit(X_train, y_train)

In [8]:
Path("../models").mkdir(exist_ok=True)

joblib.dump(
    final_pipeline,
    "../models/churn_pipeline.joblib"
)

['../models/churn_pipeline.joblib']

In [9]:
loaded_pipeline = joblib.load(
    "../models/churn_pipeline.joblib"
)

original_proba = final_pipeline.predict_proba(X_test)[:, 1]
loaded_proba = loaded_pipeline.predict_proba(X_test)[:, 1]

np.allclose(original_proba, loaded_proba)

True

The serialized pipeline produces the same probabilities after loading, confirming that the preprocessing and model were saved correctly.